In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel, AutoConfig, DataCollatorWithPadding
from torch.utils.data import Dataset, DataLoader
from transformers.utils import logging

In [2]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-small'
    tokenizer = None
    max_length = 512
    batch_size = 16
    epochs = 10
    learning_rate = 5e-5
    weight_decay = 0.01
    task = 'classification'

In [ ]:
#Quadratic Weighted Kappa score
def get_score(y_true, y_pred):
    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

In [ ]:
#Sets random to 42
def seed_everything(seed: int=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
seed_everything(42)

In [ ]:
path = kagglehub.competition_download(cfg.competition)

In [ ]:
train_df = pd.read_csv(f'{path}/train.csv')

In [ ]:
#Label y starting from 0 so classifier works
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [ ]:
#Cross Validation
train_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['score'])):
    train_df.loc[valid_idx, 'fold'] = fold

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [ ]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.long)
        return item


In [ ]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [ ]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        self.classifier = nn.Linear(self.model.config.hidden_size, 6)

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        # #CLS Pooling
        # cls_output = output.last_hidden_state[:, 0, :].float()
        # logits = self.classifier(cls_output)
        #Mean Pooling
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        logits = self.classifier(mp_output)
        return logits


In [ ]:
print(torch.backends.mps.is_available())
device = torch.device('mps')
#For CUDA:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Global Debug variable for testing
DEBUG = True

In [ ]:
#Store overall best QWK:
fold_QWK = []

#CV Pipeline
for fold in range(5):
    print(f"--------FOLD {fold}--------")

    train_fold = train_df[train_df['fold']!=fold] #Train data are the not current folds
    valid_fold = train_df[train_df['fold']==fold] #Validation is the current fold

    #Create samples for quick testing
    if DEBUG:
        train_df_used = train_fold.sample(frac=0.02, random_state=42)
        valid_df_used = valid_fold #.sample(frac=0.01, random_state=42)
    else:
        train_df_used = train_fold
        valid_df_used = valid_fold
    
    #Create Dataset and DataLoader objects
    train_dataset = EssayDataset(train_df_used, tokenizer)
    valid_dataset = EssayDataset(valid_df_used, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

    #Create model object
    logging.set_verbosity_error()
    deberta_model = EssayModel().to(device).float()
    
    #Create Optimizer
    optimizer = torch.optim.AdamW(
        deberta_model.parameters(), 
        lr=cfg.learning_rate,
        eps=1e-6,
        weight_decay=cfg.weight_decay
        )
   
    #Create Scheduler for changing LR
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    #Define Loss Function
    loss_fn = nn.CrossEntropyLoss()

    #Store best QWK, start at a negative value
    best_kappa = -999

    #Store overall loss
    overall_train_loss = 0
    overall_valid_loss = 0

    #Train Model Pipeline
    for epoch in range(cfg.epochs):
        print(f'----EPOCH {epoch+1}/{cfg.epochs}----')

        #Train
        deberta_model.train()
        train_loss = 0; train_correct = 0; train_total = 0

        for batch in train_loader:
            optimizer.zero_grad()
            labels = batch['labels'].to(device)
            logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = loss_fn(logits, labels)
            
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            train_loss += loss.item()
            overall_train_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                deberta_model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = train_correct / train_total


        #Validation
        deberta_model.eval()
        valid_loss = 0; valid_correct = 0; valid_total = 0
        all_labels = []; all_preds = []

        with torch.no_grad():
            for batch in valid_loader:
                labels = batch['labels'].to(device)
                logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
                loss = loss_fn(logits, labels)
                valid_loss += loss.item()
                overall_valid_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                valid_correct += (preds==labels).sum().item()
                valid_total += labels.size(0)

        avg_valid_loss = valid_loss/len(valid_loader)
        valid_accuracy = valid_correct/valid_total
        kappa = get_score(all_labels, all_preds)
        if kappa > best_kappa:
            best_kappa = kappa
            torch.save(
                deberta_model.state_dict(),
                f'best_deberta_model_fold_{fold}.pth'
            )
            print(f'Saved new-best model: QWK {best_kappa:.4f}')
        
        #Summary
        print(f'Train Loss: {avg_train_loss:.4f}')
        print(f'Train Accuracy: {train_accuracy:.4f}')
        print(f'Valid Loss: {avg_valid_loss:.4f}')
        print(f'Valid Accuracy: {valid_accuracy:.4f}')
        print(f'Current Epoch QWK: {kappa:.4f}')
        print(f"Overall Fold's Best QWK: {best_kappa:.4f}")
        print(f'LR: {scheduler.get_last_lr()[0]:.2e}\n')
    
    #Store best QWK for that fold
    fold_QWK.append(best_kappa)
    print(f"Fold {fold}'s best QWK: {best_kappa}")

    #Print overall loss
    print('Overall average training loss:', overall_train_loss/(cfg.epochs*len(train_loader)))
    print('Overall average validation loss:', overall_valid_loss/(cfg.epochs*len(valid_loader)))

    
#Mean QWK Across all folds
print('---------QWK SUMMARY---------')
print('Mean QWK Across 5 Folds:', np.mean(fold_QWK))
print(f'Best QWK: {np.max(fold_QWK)} | Fold: {np.argmax(fold_QWK)}')

In [ ]:
#Save configs to json
import json
config = {
    'competition': cfg.competition,
    'checkpoint': cfg.checkpoint,
    'batch_size': cfg.batch_size,
    'max_length': cfg.max_length,
    'learning_rate': cfg.learning_rate,
    'weight_decay': 0.01,
    'epochs': cfg.epochs,
    'num_labels': 1,
    'task': cfg.task,
    'best_fold': int(np.argmax(fold_QWK))
}
with open('config.json', 'w') as f:
    json.dump(config, f, indent=4)